# Learner Activity: MANOVA Service Tier Comparison

Complete the missing code to test whether service tier is associated with the combined outcomes of revenue growth and retention score.


## Setup


In [1]:
from pathlib import Path

import pandas as pd
import plotly.express as px
from scipy.stats import f_oneway
from statsmodels.multivariate.manova import MANOVA


ACTIVITY_DATA_DIR = Path("Activity Data")

DATA_FILE = ACTIVITY_DATA_DIR / "manova-sample-data.csv"
if not DATA_FILE.exists():
    raise FileNotFoundError(f"Missing data file: {DATA_FILE}")


## 1. Load And Inspect The Data


In [2]:
# Load the MANOVA dataset from DATA_FILE.
# Use pd.read_csv and pass DATA_FILE inside the parentheses.
manova_df = pd.read_csv(ACTIVITY_DATA_DIR / "manova-sample-data.csv")

# Display the first few rows so you can check the column names.
manova_df.head()


,account_name,service_tier,revenue_growth_pct,retention_score
0,BAS-01,Basic,6.7,76
1,BAS-02,Basic,6.8,77
2,BAS-03,Basic,7.0,76
3,BAS-04,Basic,6.2,77
4,BAS-05,Basic,6.6,77


In [3]:
# Identify the grouping column and the two outcome columns.
group_column = "service_tier"
outcome_columns = [
    "revenue_growth_pct",
    "retention_score",
]

# Calculate average outcome values by service tier.
# Group by group_column, then select outcome_columns.
tier_means = manova_df.groupby(group_column)[outcome_columns].mean().round(2)
tier_means


,revenue_growth_pct,retention_score
service_tier,,
Basic,6.55,77.00
Enterprise,9.55,84.25
Professional,12.46,90.12


## 2. Visualize The Two Outcomes

Create a scatter plot with revenue growth on one axis, retention score on the other, and service tier as the grouping variable.


In [4]:
# Create a scatter plot with one outcome on each axis.
# Use service tier for color and symbol so groups are easy to compare.
fig = px.scatter(
    manova_df,
    x="revenue_growth_pct",
    y="retention_score",
    color="service_tier",
    symbol="service_tier",
    hover_data=["account_name"],
    title="MANOVA: Revenue Growth and Retention by Service Tier",
)
fig.update_layout(template="plotly_white")
fig.show()


## 3. Fit The MANOVA Model

Use the formula pattern `outcome_1 + outcome_2 ~ group_column`.


In [5]:
# Use the MANOVA formula pattern: outcome_1 + outcome_2 ~ group_column.
# For this dataset, compare revenue_growth_pct and retention_score by service_tier.
formula = "revenue_growth_pct + retention_score ~ service_tier"

manova = MANOVA.from_formula(
    formula,
    data=manova_df,
)
manova_result = manova.mv_test()
print(manova_result)


                    Multivariate linear model
                                                                  
------------------------------------------------------------------
       Intercept          Value   Num DF  Den DF  F Value   Pr > F
------------------------------------------------------------------
          Wilks' lambda    0.0006 2.0000 20.0000 16832.2507 0.0000
         Pillai's trace    0.9994 2.0000 20.0000 16832.2507 0.0000
 Hotelling-Lawley trace 1683.2251 2.0000 20.0000 16832.2507 0.0000
    Roy's greatest root 1683.2251 2.0000 20.0000 16832.2507 0.0000
------------------------------------------------------------------
                                                                  
------------------------------------------------------------------
         service_tier       Value  Num DF  Den DF  F Value  Pr > F
------------------------------------------------------------------
             Wilks' lambda  0.0098 4.0000 40.0000   90.7746 0.0000
            Pill

## 4. Interpretation

Write 2-3 sentences: Are the service-tier test p-values below 0.05? Which tier appears strongest across both outcomes? Why is MANOVA useful here instead of two separate first-pass tests?

ANSWER: Yes, the MANOVA result show that p-values for service-tier are below 0.05. This indicates statistically significant differences among service tiers. The Professional tier appears strongest across both outcomes because it has the highest average revenue growth percentages and retention scores in the dataset. MANOVA is useful here instead of two separate first-pass tests because it evaluates both related business outcomes simultaneously, accounting for their relationship while reducing the risk of inflated false-positive results from running separate tests independently.